# Phase 5.2 — Matching engine and data-contract hardening

## Goal

Install and execute the additive hardening layer on the signed Phase 4/5
artifacts. This run normalizes the reference contract, applies authorization,
filters, exclusions, hybrid retrieval, evidence reranking, complete-`MUST`
eligibility, duplicate control, citation integrity checks, and safe spreadsheet
serialization.

The existing Phase 4–8 outputs remain unchanged. The packaged Phase 6 offer is
a redacted engineering fixture; Phase 5.1 expert labels remain mandatory before
any production relevance claim or retrieval-model promotion.

### Expected handoff

The final cell prints the Phase 5.2 status, every quality-gate check, the V1/V2
technical comparison, and the exact output folder to review.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile

PROJECT_FOLDER_NAME = "Devoteam_AI_CLEAN_PIPELINE"
PROJECT_PARENT_FOLDER_NAME = "Devoteam internship"
PACKAGE_FILENAME = "PHASE_5_2_MATCHING_ENGINE_HARDENING_PACKAGE.zip"
PACKAGE_SHA256 = "1486ce3436366aa8c47e91d869a962d52545f8ae2e3cf59949a43ba00c095054"
PACKAGE_MANIFEST_SHA256 = "4f8d323e808e978d2c1b57281e28f52ed782ccbae4c25aa830fa5d9107a14e47"
SNAPSHOT_ID = "20260714T154731Z_129ff982c8"
PHASE5_RUN_NAME = "phase5_hybrid_retrieval_v1"
PHASE5_2_RUN_NAME = "phase5_2_matching_hardening_v1"

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

print("Phase 5.2 contract loaded: additive hardening; source mutation disabled.")

## 1. Locate the clean project

The Drive path is explicit so Colab does not perform a slow recursive search.
Local validation may set `DEVOTEAM_PROJECT_ROOT` and
`DEVOTEAM_PHASE5_2_PACKAGE`.

In [ ]:
override = os.environ.get("DEVOTEAM_PROJECT_ROOT")
if override:
    PROJECT_ROOT = Path(override).resolve()
else:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    PROJECT_ROOT = (
        Path("/content/drive/MyDrive")
        / PROJECT_PARENT_FOLDER_NAME
        / PROJECT_FOLDER_NAME
    ).resolve()
    assert PROJECT_ROOT.is_dir(), f"Clean project folder not found: {PROJECT_ROOT}"
    assert (PROJECT_ROOT / "config" / "project.yaml").exists(), "Project configuration is missing"

PACKAGE_PATH = Path(
    os.environ.get("DEVOTEAM_PHASE5_2_PACKAGE", PROJECT_ROOT / PACKAGE_FILENAME)
).resolve()
assert PROJECT_ROOT.name == PROJECT_FOLDER_NAME, PROJECT_ROOT
assert PACKAGE_PATH.exists(), f"Missing package: {PACKAGE_PATH}"
print(f"Project root: {PROJECT_ROOT}")
print(f"Signed package: {PACKAGE_PATH}")

## 2. Verify and install the signed additive overlay

Only manifest-listed files are accepted. Existing identical files are skipped;
any conflicting file stops the run. The Phase 5.2 tests run by default. Set
`DEVOTEAM_RUN_FULL_REGRESSION=1` to rerun all project tests.

In [ ]:
assert file_sha256(PACKAGE_PATH) == PACKAGE_SHA256, "Phase 5.2 package hash mismatch"
with zipfile.ZipFile(PACKAGE_PATH) as archive:
    names = archive.namelist()
    assert "PHASE_5_2_PACKAGE_MANIFEST.json" in names
    manifest_bytes = archive.read("PHASE_5_2_PACKAGE_MANIFEST.json")
    assert hashlib.sha256(manifest_bytes).hexdigest() == PACKAGE_MANIFEST_SHA256
    package_manifest = json.loads(manifest_bytes)
    allowed = set(package_manifest["files"]) | {"PHASE_5_2_PACKAGE_MANIFEST.json"}
    assert set(names) == allowed, "Package contains undeclared files"
    installed = skipped = 0
    for name in names:
        target = (PROJECT_ROOT / name).resolve()
        assert target == PROJECT_ROOT or PROJECT_ROOT in target.parents, name
        data = archive.read(name)
        if name != "PHASE_5_2_PACKAGE_MANIFEST.json":
            expected = package_manifest["files"][name]
            assert hashlib.sha256(data).hexdigest() == expected["sha256"]
            assert len(data) == int(expected["size_bytes"])
        if target.exists():
            assert target.read_bytes() == data, f"Conflicting existing Phase 5.2 file: {name}"
            skipped += 1
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(data)
            installed += 1

requirements = PROJECT_ROOT / "requirements" / "phase5_2.txt"
if os.environ.get("DEVOTEAM_SKIP_PIP") != "1":
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)],
        check=True,
    )

environment = os.environ.copy()
environment["PYTHONPATH"] = str(PROJECT_ROOT / "src") + os.pathsep + environment.get("PYTHONPATH", "")
test_target = PROJECT_ROOT / (
    "tests"
    if os.environ.get("DEVOTEAM_RUN_FULL_REGRESSION") == "1"
    else "tests/test_phase5_2_matching.py"
)
tests = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(test_target)],
    cwd=PROJECT_ROOT,
    env=environment,
    text=True,
    capture_output=True,
)
print(tests.stdout[-5000:])
assert tests.returncode == 0, tests.stderr[-5000:]
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Package verified: installed={installed}, unchanged={skipped}")

## 3. Resolve and verify signed inputs

Phase 5.2 reuses the Phase 5 BM25 index and passage embeddings. It never rebuilds
or edits them. If an already-complete Phase 5.2 run exists, this notebook verifies
it without loading the embedding model.

In [ ]:
from devoteam_reference_ai.phase5_retrieval import E5EmbeddingAdapter, load_phase5_config
from devoteam_reference_ai.phase5_2_matching import (
    load_phase5_2_config,
    run_phase5_2,
    verify_phase5_2,
)

PHASE5_CONFIG = load_phase5_config(PROJECT_ROOT / "config" / "phase5_retrieval.yaml")
PHASE5_2_CONFIG = load_phase5_2_config(
    PROJECT_ROOT / "config" / "phase5_2_matching_hardening.yaml"
)
PHASE5_ROOT = (
    PROJECT_ROOT / "data" / "indexes" / SNAPSHOT_ID / PHASE5_RUN_NAME
)
RUN_ROOT = (
    PROJECT_ROOT / "data" / "indexes" / SNAPSHOT_ID / PHASE5_2_RUN_NAME
)
assert PHASE5_ROOT.is_dir(), PHASE5_ROOT
print(f"Signed Phase 5 input: {PHASE5_ROOT}")
print(f"Phase 5.2 output: {RUN_ROOT}")

## 4. Run the hardening pipeline

On the first run, the pinned multilingual E5 model is loaded locally to embed
only the offer requirements and bootstrap probes. Existing passage embeddings
are reused. External LLM and external embedding API calls remain disabled.

In [ ]:
success_marker = RUN_ROOT / PHASE5_2_CONFIG["output"]["success_marker"]
if success_marker.exists():
    RESULT = verify_phase5_2(RUN_ROOT, PHASE5_2_CONFIG)
    RESULT["run_root"] = str(RUN_ROOT)
    RESULT["resumed"] = True
    print("Existing Phase 5.2 output verified; model loading skipped.")
else:
    adapter = E5EmbeddingAdapter(PHASE5_CONFIG)
    RESULT = run_phase5_2(
        PROJECT_ROOT,
        PROJECT_ROOT / "config" / "phase5_2_matching_hardening.yaml",
        phase5_config=PHASE5_CONFIG,
        embedding_adapter=adapter,
        progress=print,
    )

MANIFEST = RESULT["manifest"]
QUALITY = RESULT["quality_gate"]
assert MANIFEST["status"] == "TECHNICAL_PASS_SAMPLE_ONLY"
assert QUALITY["technical_gate"] == "PASS"
assert MANIFEST["production_quality_claim_allowed"] is False
assert MANIFEST["production_promotion_status"] == "BLOCKED_PENDING_PHASE_5_1_EXPERT_EVALUATION"
print("PHASE 5.2 MATCHING HARDENING: PASS")

## 5. Review the result and decision boundary

The checks below prove control correctness and reproducibility. They do not
replace expert relevance labels or human citation-support review.

In [ ]:
comparison = json.loads((RUN_ROOT / "v1_v2_comparison.json").read_text(encoding="utf-8"))
print(f"Status: {MANIFEST['status']}")
print(f"Normalized reference rows: {MANIFEST['source_references']}")
print(f"Base shortlist-eligible references: {MANIFEST['base_shortlist_eligible_references']}")
print(f"Retrieval mode: {MANIFEST['retrieval_mode']}")
print(f"Content requirements searched: {MANIFEST['content_requirements']}")
print(f"Eligibility policies separated: {MANIFEST['policy_requirements']}")
print(f"Recommendations / ineligible diagnostics: {MANIFEST['recommendations']} / {MANIFEST['ineligible_candidates']}")
print(f"Citation completeness / integrity: {MANIFEST['citation_completeness']:.0%} / {MANIFEST['citation_integrity']:.0%}")
print(f"Citation correctness: {MANIFEST['citation_correctness_status']}")
print("\nQuality gates:")
for name, check in QUALITY["checks"].items():
    print(f"  {'PASS' if check['passed'] else 'FAIL'}  {name}: {check['value']:.3f} >= {check['threshold']:.3f}")
print("\nV1 → V2 technical controls:")
print(f"  Retrieval: {comparison['v1']['retrieval_mode']} → {comparison['v2']['retrieval_mode']}")
print(f"  Shortlisted rows missing MUST: {comparison['v1']['recommendations_missing_must']} → {comparison['v2']['recommendations_missing_must']}")
print(f"  Top-reference overlap: {comparison['top_reference_overlap']}")
print(f"\nOutput folder: {RUN_ROOT}")
print("IMPORTANT: Phase 5.1 remains the production relevance-promotion gate.")

## Next step

After this notebook prints `PHASE 5.2 MATCHING HARDENING: PASS`, return with the
final output (or a screenshot). We will inspect the real E5 fusion sweep,
recommendations, excluded candidates, filter audit, and citation-audit workbook.
Only after that review will we process the first real authorized offer using the
hardened matcher.